# 第 8 章 · 记忆（短期：SessionService）

原书给了三种会话后端的**构造代码**，本文件保持原样。

两处必要适配：`DatabaseSessionService` 的 URL 改成异步驱动 `sqlite+aiosqlite`；
`VertexAiSessionService` 需要 GCP，包在 `try` 里。

kernel 选 `Python (agentic-adk)`。三种后端的速度实测、state 前缀落到哪张表、
与 `MemoryService` 的分工见 `../../my_notes/ch08_memory.md`。


In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）
from common import quiet_adk_sync_warnings, require_adk
quiet_adk_sync_warnings()
require_adk()      # 【适配】内核选错时给出可操作提示

# Example: Using InMemorySessionService
# This is suitable for local development and testing where data persistence
# across application restarts is not required.
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()
print("InMemorySessionService 已构造:", type(session_service).__name__)


InMemorySessionService 已构造: InMemorySessionService


In [2]:

# Example: Using DatabaseSessionService
# This is suitable for production or development requiring persistent storage.
# You need to configure a database URL (e.g., for SQLite, PostgreSQL, etc.).
# Requires: pip install google-adk[sqlalchemy] and a database driver (e.g., psycopg2 for PostgreSQL)
from google.adk.sessions import DatabaseSessionService

# 【适配】书里写的是 sqlite:///./my_agent_data.db；该服务要求**异步**驱动，
#   同步 URL 会报 "resolves to a synchronous driver"，故改为 sqlite+aiosqlite
db_url = "sqlite+aiosqlite:///./my_agent_data.db"
session_service = DatabaseSessionService(db_url=db_url)
print("DatabaseSessionService 已构造，库文件:", db_url.split("///")[-1])


DatabaseSessionService 已构造，库文件: ./my_agent_data.db


In [3]:

# Example: Using VertexAiSessionService
# This is suitable for scalable production on Google Cloud Platform, leveraging
# Vertex AI infrastructure for session management.
# Requires: pip install google-adk[vertexai] and GCP setup/authentication
from google.adk.sessions import VertexAiSessionService

PROJECT_ID = "your-gcp-project-id"        # Replace with your GCP project ID
LOCATION = "us-central1"                  # Replace with your desired GCP location
# The app_name used with this service should correspond to the Reasoning Engine ID or name
REASONING_ENGINE_APP_NAME = "projects/your-gcp-project-id/locations/us-central1/reasoningEngines/your-engine-id"

# 【适配】构造需要 GCP 凭据，本机包在 try 里，保持书里的写法不变
try:
    session_service = VertexAiSessionService(project=PROJECT_ID, location=LOCATION)
    print("VertexAiSessionService 已构造")
except Exception as e:      # noqa: BLE001
    print("VertexAiSessionService 不可用（需 GCP 项目 + 认证）:", type(e).__name__)

# When using this service, pass REASONING_ENGINE_APP_NAME to service methods:
# session_service.create_session(app_name=REASONING_ENGINE_APP_NAME, ...)
# session_service.get_session(app_name=REASONING_ENGINE_APP_NAME, ...)
# session_service.append_event(session, event, app_name=REASONING_ENGINE_APP_NAME)
# session_service.delete_session(app_name=REASONING_ENGINE_APP_NAME, ...)


VertexAiSessionService 不可用（需 GCP 项目 + 认证）: ImportError
